# SDC-2022 — Collection Track Maps

Maps the **ground-truth trajectory** of every 2022 session used in the multipath pipeline, so the routes driven can be inspected against the multipath findings. Each session's reference track comes from its `ground_truth.csv` (`LatitudeDegrees` / `LongitudeDegrees`).

Sessions are grouped by geographic area (Mountain View, San Jose, Los Angeles). Green marker = start, red marker = end.

## 1. Setup

Uses **folium** for interactive Leaflet maps (same library as `00_coordinate_maps.ipynb`). If it is not installed in the kernel, the next cell installs it.

In [ ]:
try:
    import folium
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'folium'])
    import folium

import os
import glob
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../..'))
RAW_DIR  = os.path.join(BASE_DIR, 'data/01_raw/sdc2023/train')
print('folium', folium.__version__)
print('BASE_DIR =', BASE_DIR)

## 2. Which sessions were used?

Re-derives the exact set of sessions that fed the classifier: every 2022 session containing at least one device that reports `MultipathIndicator = 1`.

In [ ]:
gnss_files = sorted(glob.glob(os.path.join(RAW_DIR, '2022-*', '*', 'device_gnss.csv')))

used_sessions = set()
for f in gnss_files:
    parts = f.replace('\\', '/').split('/')
    sess = parts[-3]
    mp = pd.read_csv(f, usecols=['MultipathIndicator'])['MultipathIndicator']
    if (mp == 1).any():
        used_sessions.add(sess)

used_sessions = sorted(used_sessions)
print(f'{len(used_sessions)} sessions with multipath:')
for s in used_sessions:
    print('  ', s)

## 3. Load Ground-Truth Tracks

For each used session, load one representative `ground_truth.csv` (all devices in a session follow the same reference trajectory) and downsample for a light-weight polyline.

In [ ]:
def load_track(session):
    gts = sorted(glob.glob(os.path.join(RAW_DIR, session, '*', 'ground_truth.csv')))
    if not gts:
        return None
    g = pd.read_csv(gts[0])
    g = g.dropna(subset=['LatitudeDegrees', 'LongitudeDegrees'])
    g = g[(g['LatitudeDegrees'].between(-90, 90)) & (g['LongitudeDegrees'].between(-180, 180))]
    if g.empty:
        return None
    step = max(1, len(g) // 2000)   # cap ~2000 pts per track
    pts = list(zip(g['LatitudeDegrees'][::step], g['LongitudeDegrees'][::step]))
    return pts

tracks = {}
for s in used_sessions:
    pts = load_track(s)
    if pts:
        tracks[s] = pts
        clat, clon = np.mean([p[0] for p in pts]), np.mean([p[1] for p in pts])
        print(f'{s:38} {len(pts):>5} pts   center=({clat:.4f}, {clon:.4f})')
    else:
        print(f'{s:38} no ground-truth coordinates')

## 4. Overview Map — All Sessions

Every used session on one interactive map, colour-cycled, with a satellite-imagery layer available from the layer control (top-right).

In [ ]:
COLORS = ['blue', 'red', 'green', 'orange', 'purple', 'darkblue', 'darkred',
          'cadetblue', 'darkgreen', 'black', 'pink', 'darkpurple', 'lightblue',
          'beige', 'lightgreen', 'gray', 'lightred']

def build_map(session_pts, zoom=11):
    all_pts = [p for pts in session_pts.values() for p in pts]
    center = [np.mean([p[0] for p in all_pts]), np.mean([p[1] for p in all_pts])]
    m = folium.Map(location=center, zoom_start=zoom, tiles='cartodbpositron')
    folium.TileLayer(
        'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Esri Satellite', overlay=False, control=True).add_to(m)
    for i, (label, pts) in enumerate(session_pts.items()):
        color = COLORS[i % len(COLORS)]
        folium.PolyLine(pts, color=color, weight=4, opacity=0.8, tooltip=label).add_to(m)
        folium.Marker(pts[0],  popup=f'Start: {label}', icon=folium.Icon(color='green', icon='play')).add_to(m)
        folium.Marker(pts[-1], popup=f'End: {label}',   icon=folium.Icon(color=color,  icon='stop')).add_to(m)
    folium.LayerControl().add_to(m)
    return m

overview = build_map(tracks, zoom=9)
display(overview)

## 5. Regional Maps

The overview spans the whole Bay Area, so each geographic cluster is also shown zoomed-in. Sessions are grouped by the location tag in their folder name.

In [ ]:
def region_of(session):
    if 'mtv' in session: return 'Mountain View'
    if 'sjc' in session: return 'San Jose'
    if 'lax' in session: return 'Los Angeles'
    return 'Other'

regions = {}
for s, pts in tracks.items():
    regions.setdefault(region_of(s), {})[s] = pts

for region, sess_pts in regions.items():
    print(f'\n### {region}  —  {len(sess_pts)} session(s)')
    display(build_map(sess_pts, zoom=12))

## 6. Session Summary Table

Start/end coordinates and point counts for every mapped session.

In [ ]:
rows = []
for s, pts in tracks.items():
    rows.append({'session': s, 'region': region_of(s), 'n_points': len(pts),
                 'start_lat': round(pts[0][0], 5), 'start_lon': round(pts[0][1], 5),
                 'end_lat': round(pts[-1][0], 5), 'end_lon': round(pts[-1][1], 5)})
summary = pd.DataFrame(rows).sort_values(['region', 'session']).reset_index(drop=True)
summary